# IncomeLens ML — Walkthrough
**Goal:** Predict whether a person earns `>50K` or `<=50K` per year based on census data.

This notebook walks through the exact ML pipeline used in the IncomeLens project — from raw data to a deployed prediction API.

---
## Step 1 — Install & Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix
)

print('All imports OK')

---
## Step 2 — Load the Dataset

We use the **UCI Adult Census dataset** — 32,561 rows of real census records.

Each row represents one person with 14 demographic + employment features.
The target column is `income`: either `<=50K` or `>50K`.

> Upload `adult.csv` using the Colab file panel (folder icon on the left), or mount Google Drive.

In [ ]:
df = pd.read_csv('adult.csv')
print(f'Shape: {df.shape}')   # rows x columns
df.head()

---
## Step 3 — Class Distribution (The Imbalance Problem)

Before touching any model, we check how balanced the target classes are.

**Why this matters:** If one class dominates, a model can cheat — predicting the majority class every time and still scoring high accuracy. We need to know this upfront to choose the right metric.

In [ ]:
counts = df['income'].str.strip().value_counts()
pct    = df['income'].str.strip().value_counts(normalize=True) * 100

print('Class counts:')
for label in counts.index:
    print(f'  {label:>6} : {counts[label]:>6}  ({pct[label]:.1f}%)')

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(counts.index, counts.values, color=['#4C9BE8', '#E8874C'], edgecolor='white')
ax.bar_label(bars, labels=[f'{p:.1f}%' for p in pct.values], padding=4, fontsize=11)
ax.set_title('Income Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()

print()
print('KEY INSIGHT: ~75% of samples are <=50K.')
print('A model that always predicts <=50K gets 75% accuracy — but is useless.')
print('=> We must use F1 Score, not accuracy, as our evaluation metric.')

---
## Step 4 — Data Cleaning

The dataset uses `?` as a placeholder for missing values (common in older UCI datasets).
We strip whitespace and convert `?` to proper `NaN` so sklearn can handle it.

In [ ]:
print('Missing values BEFORE cleaning:')
print(df.isin(['?', ' ?', '? ']).sum()[lambda s: s > 0])

# Clean: strip whitespace + replace '?' with NaN
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({'?': np.nan, 'nan': np.nan})

print()
print('Missing values AFTER cleaning (as NaN):')
missing = df.isnull().sum()
print(missing[missing > 0])
print()
print('These will be handled inside the sklearn Pipeline using SimpleImputer — NOT dropped.')

---
## Step 5 — Train / Test Split with Stratification

We split 80% train / 20% test.

**`stratify=y` is critical** — it ensures both splits have the same ~75/25 class ratio.
Without it, by random chance the test set could be unrepresentative, giving misleading evaluation results.

In [ ]:
TARGET       = 'income'
POSITIVE_LABEL = '>50K'

df = df.dropna(subset=[TARGET])
y_raw = df[TARGET].astype(str).str.strip()
X     = df.drop(columns=[TARGET])
y     = (y_raw == POSITIVE_LABEL).astype(int)   # 1 = >50K, 0 = <=50K

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {len(X_train):>6} rows')
print(f'Test  size : {len(X_test):>6} rows')
print()
print(f'Train class ratio  =>  <=50K: {(y_train==0).mean()*100:.1f}%  |  >50K: {(y_train==1).mean()*100:.1f}%')
print(f'Test  class ratio  =>  <=50K: {(y_test==0).mean()*100:.1f}%  |  >50K: {(y_test==1).mean()*100:.1f}%')
print()
print('Both splits mirror the original 75/25 ratio — stratification worked.')

---
## Step 6 — Build the sklearn Pipeline (Preventing Data Leakage)

This is the most important engineering decision in the project.

**What is data leakage?**
If you fit the imputer or encoder on the *entire dataset* before splitting, the test set "leaks" statistical information into training. Your model appears better than it really is.

**How Pipeline prevents it:**
- `pipeline.fit(X_train, y_train)` — imputer and encoder learn ONLY from training data
- `pipeline.predict(X_test)` — the same learned transforms are *applied* to test data without refitting

The pipeline also handles two feature types differently:
- **Numeric** (age, capital.gain, hours.per.week …) → impute with **median** (robust to outliers)
- **Categorical** (workclass, occupation, education …) → impute with **most frequent**, then **one-hot encode**

In [ ]:
numeric_cols     = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print('Numeric features    :', numeric_cols)
print('Categorical features:', categorical_cols)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore')),
])

# handle_unknown='ignore' -> if a new unseen category arrives at inference time,
# it outputs all zeros instead of crashing

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,     numeric_cols),
    ('cat', categorical_transformer, categorical_cols),
])

print()
print('Preprocessor built. It will be fitted ONLY on X_train inside the pipeline.')

---
## Step 7 — Model Selection via GridSearchCV

We don't just pick one model — we compare two families:

| Model | Strength | Weakness |
|---|---|---|
| Logistic Regression | Fast, interpretable, works well linearly | Assumes linear decision boundary |
| Random Forest | Handles non-linearity, mixed types, gives feature importance | Slower, less interpretable |

**GridSearchCV** tries all combinations of hyperparameters and picks the best using **3-fold cross-validation**.
We score on **F1** (not accuracy) because of the class imbalance we saw in Step 3.

> This cell will take a few minutes to run.

In [ ]:
# --- Logistic Regression pipeline ---
logreg_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
    # class_weight='balanced' compensates for imbalance by weighting minority class higher
])

logreg_grid = {
    'clf__C'      : [0.1, 1.0, 3.0],   # regularization strength (lower = more regularization)
    'clf__penalty': ['l2'],
    'clf__solver' : ['lbfgs'],
}

# --- Random Forest pipeline ---
rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('clf', RandomForestClassifier(random_state=42)),
])

rf_grid = {
    'clf__n_estimators'    : [200, 400],        # number of trees
    'clf__max_depth'       : [None, 10, 20],    # None = grow until pure leaves
    'clf__min_samples_split': [2, 5],           # minimum samples to split a node
}

print('Training Logistic Regression with GridSearchCV...')
gs_lr = GridSearchCV(logreg_pipeline, logreg_grid, scoring='f1', cv=3, n_jobs=-1)
gs_lr.fit(X_train, y_train)
print(f'  Best params : {gs_lr.best_params_}')
print(f'  Best CV F1  : {gs_lr.best_score_:.4f}')

print()
print('Training Random Forest with GridSearchCV...')
gs_rf = GridSearchCV(rf_pipeline, rf_grid, scoring='f1', cv=3, n_jobs=-1)
gs_rf.fit(X_train, y_train)
print(f'  Best params : {gs_rf.best_params_}')
print(f'  Best CV F1  : {gs_rf.best_score_:.4f}')

---
## Step 8 — Evaluate on Held-Out Test Set

Cross-validation picked the best hyperparameters, but the **final score must come from the test set** — data the model has never seen at all.

We compare both winners on the same test set to pick the final model.

In [ ]:
best_lr = gs_lr.best_estimator_
best_rf = gs_rf.best_estimator_

lr_pred = best_lr.predict(X_test)
rf_pred = best_rf.predict(X_test)

lr_f1 = f1_score(y_test, lr_pred)
rf_f1 = f1_score(y_test, rf_pred)

print(f'Test F1 — Logistic Regression : {lr_f1:.4f}')
print(f'Test F1 — Random Forest       : {rf_f1:.4f}')
print()

winner = 'Random Forest' if rf_f1 >= lr_f1 else 'Logistic Regression'
best   = best_rf         if rf_f1 >= lr_f1 else best_lr
print(f'Winner: {winner}')

# Bar chart comparison
fig, ax = plt.subplots(figsize=(5, 3))
models = ['Logistic Regression', 'Random Forest']
scores = [lr_f1, rf_f1]
colors = ['#4C9BE8', '#E8874C']
bars = ax.bar(models, scores, color=colors, edgecolor='white')
ax.bar_label(bars, labels=[f'{s:.4f}' for s in scores], padding=4, fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_ylabel('F1 Score (Test Set)')
ax.set_title('Model Comparison — Test F1', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 9 — Classification Report

F1 alone doesn't tell the full story. The classification report breaks it down by class:

- **Precision** — of all predictions of >50K, how many were correct?
- **Recall** — of all actual >50K people, how many did we catch?
- **F1** — harmonic mean of precision and recall

This lets us see if the model is biased toward predicting one class more than the other.

In [ ]:
best_pred = best.predict(X_test)

print('Classification Report — Winning Model:')
print(classification_report(y_test, best_pred, target_names=['<=50K', '>50K']))

# Confusion matrix
cm = confusion_matrix(y_test, best_pred)
fig, ax = plt.subplots(figsize=(4, 3.5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['<=50K', '>50K'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('Reading the matrix:')
print('  Top-left  (TN): Correctly predicted <=50K')
print('  Top-right (FP): Predicted >50K but actually <=50K')
print('  Bot-left  (FN): Predicted <=50K but actually >50K  <- the costly errors')
print('  Bot-right (TP): Correctly predicted >50K')

---
## Step 10 — Feature Importance

**What drives the prediction?**

Random Forest gives us `feature_importances_` — the average reduction in Gini impurity across all trees when splitting on each feature.
A higher value = that feature is more useful for making the right split.

After one-hot encoding, each category becomes its own binary feature, so we see things like `marital.status_Married-civ-spouse` as a standalone predictor.

In [ ]:
pre = best.named_steps['preprocess']
clf = best.named_steps['clf']

feature_names = pre.get_feature_names_out()
importances   = clf.feature_importances_

fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(12).reset_index(drop=True)

# Clean up prefixes added by ColumnTransformer
fi_df['feature'] = fi_df['feature'].str.replace(r'^(num__|cat__)', '', regex=True)

print('Top 12 most important features:')
print(fi_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 5))
colors = plt.cm.Blues_r(np.linspace(0.2, 0.8, len(fi_df)))
bars = ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 12 Predictors of Income >50K', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
plt.tight_layout()
plt.show()

print()
print('Key takeaways:')
print('  - capital.gain is the strongest signal — investment income strongly predicts high earners')
print('  - marital.status_Married is 2nd — correlated with older, more established earners')
print('  - education.num and age follow — years of education and career stage matter')
print('  - hours.per.week — more hours = more income, but weaker than expected')

---
## Step 11 — Live Prediction (same as the app)

This is exactly what happens when a user fills the form and clicks **Predict** in the IncomeLens app.
The FastAPI backend receives the form values as a dictionary and runs them through this same pipeline.

In [ ]:
THRESHOLD = 0.5

sample = {
    'age'           : 37,
    'workclass'     : 'Private',
    'fnlwgt'        : 215646,
    'education'     : 'Bachelors',
    'education.num' : 13,
    'marital.status': 'Married-civ-spouse',
    'occupation'    : 'Prof-specialty',
    'relationship'  : 'Husband',
    'race'          : 'White',
    'sex'           : 'Male',
    'capital.gain'  : 0,
    'capital.loss'  : 0,
    'hours.per.week': 40,
    'native.country': 'United-States',
}

X_sample = pd.DataFrame([sample], columns=X.columns)

proba     = best.predict_proba(X_sample)[0, 1]
pred_int  = 1 if proba >= THRESHOLD else 0
label     = '>50K' if pred_int == 1 else '<=50K'

print('Input:')
for k, v in sample.items():
    print(f'  {k:<18}: {v}')
print()
print(f'Predicted income class : {label}')
print(f'Confidence (>50K prob) : {proba*100:.1f}%')
print(f'Decision threshold     : {THRESHOLD}')
print()
print('This is the exact output the /predict API endpoint returns to the React frontend.')

---
## Summary

| Step | What we did | Why |
|---|---|---|
| Class distribution | Checked imbalance (~75/25) | Chose F1 over accuracy |
| Data cleaning | Replaced `?` with NaN | Proper missing value handling |
| Stratified split | `stratify=y` in train_test_split | Preserve class ratio in both sets |
| sklearn Pipeline | Preprocessor + model in one object | Prevent data leakage |
| GridSearchCV | Tuned hyperparameters with 3-fold CV | Find optimal settings without touching test set |
| Model comparison | Logistic Regression vs Random Forest | Chose best by test F1 |
| Feature importance | Gini-based importances from RF | Understand what drives predictions |
| Live prediction | `predict_proba` + threshold | Same flow used in the deployed API |